In [3]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import wordnet, stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def preprocess(sentence):
    tokens = word_tokenize(sentence.lower())
    return [w for w in tokens if w.isalpha() and w not in stop_words]

def improved_lesk(sentence, target_word):
    context = set(preprocess(sentence))
    best_sense = None
    max_overlap = 0

    for synset in wordnet.synsets(target_word):
        signature = synset.definition().split()
        for ex in synset.examples():
            signature += ex.split()

        signature = set(w.lower() for w in signature)

        overlap = len(context.intersection(signature))

        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = synset

    return best_sense

def perform_wsd(sentence, target_word):
    sense = improved_lesk(sentence, target_word)

    print("\nSentence:", sentence)
    print("Word:", target_word)

    if sense:
        print("Sense:", sense.name())
        print("Definition:", sense.definition())
    else:
        print("Sense: Not Found")

text1 = "I went to the bank to deposit money"
text2 = "We had a picnic near the bank"

perform_wsd(text1, "bank")
perform_wsd(text2, "bank")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.



Sentence: I went to the bank to deposit money
Word: bank
Sense: depository_financial_institution.n.01
Definition: a financial institution that accepts deposits and channels the money into lending activities

Sentence: We had a picnic near the bank
Word: bank
Sense: bank.n.01
Definition: sloping land (especially the slope beside a body of water)


In [5]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
from nltk import word_tokenize, pos_tag
from nltk.tree import Tree
from nltk.parse import ChartParser
from nltk.grammar import CFG

text = "Sundar Pichai is the CEO of Google"

grammar = CFG.fromstring("""
S -> NP VP
VP -> VBZ NP | VBZ NP PP
NP -> NNP | NNP NNP | DT NN | DT NN PP
PP -> IN NP

NNP -> 'Sundar' | 'Pichai' | 'Google'
NN -> 'CEO'
DT -> 'the'
VBZ -> 'is'
IN -> 'of'
""")

parser = ChartParser(grammar)

def constituency_parse(sentence):
    print("\n=== Constituency Parsing ===\n")
    tokens = word_tokenize(sentence)

    for tree in parser.parse(tokens):
        print(tree)
        tree.pretty_print()

def dependency_parse(sentence):
    print("\n=== Dependency Parsing (Basic NLTK Approximation) ===\n")

    tokens = word_tokenize(sentence)
    pos_tags = pos_tag(tokens)

    # Simple rule-based dependency (approx)
    for i, (word, tag) in enumerate(pos_tags):
        if tag.startswith('NN'):
            print(f"{word} --> subject/object")
        elif tag.startswith('VB'):
            print(f"{word} --> root (verb)")
        elif tag == 'IN':
            print(f"{word} --> preposition")
        else:
            print(f"{word} --> modifier")

print("\nProcessing Small Dataset...")
constituency_parse(text)
dependency_parse(text)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.



Processing Small Dataset...

=== Constituency Parsing ===

(S
  (NP (NNP Sundar) (NNP Pichai))
  (VP (VBZ is) (NP (DT the) (NN CEO)) (PP (IN of) (NP (NNP Google)))))
                       S                        
         ______________|_______                  
        |                      VP               
        |           ___________|_______          
        |          |       |           PP       
        |          |       |        ___|____     
        NP         |       NP      |        NP  
   _____|____      |    ___|___    |        |    
 NNP        NNP   VBZ  DT      NN  IN      NNP  
  |          |     |   |       |   |        |    
Sundar     Pichai  is the     CEO  of     Google

(S
  (NP (NNP Sundar) (NNP Pichai))
  (VP (VBZ is) (NP (DT the) (NN CEO) (PP (IN of) (NP (NNP Google))))))
             S                              
         ____|_________                      
        |              VP                   
        |           ___|___                  